Explore the Discover Page
Now let's explore what's available in your workspace. The Discover page is where you browse models and datasets.

UI -> Left nav -> Discover

What to look for:

Model types available: Chat models (for generation), Embedding models (for Vector Search), Code models, etc. <br>
Model providers: Databricks-hosted (Meta Llama, etc.), third-party <br>
Task filters: Filter by task to find models for your specific need <br>


UI -> Left nav -> Marketplace<br>
What to look for:

Pre-built solutions: Agent templates, data products, ML models <br>
Datasets: Public datasets you can use for testing and development <br>
Providers: Databricks, partners, and community contributors<br>

- - **Note** Know that Discover shows workspace-local assets and Marketplace shows installable external assets. They're different UI sections.

In [0]:
# ------------------------------------------------------------------
# Setup: Install dependencies
# ------------------------------------------------------------------

%pip install openai --quiet

# Restart Python to pick up the new package
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# ------------------------------------------------------------------
# Setup: Initialize the OpenAI-compatible client for Databricks
# ------------------------------------------------------------------
# Note : here i have used my personal open AI token  which asks as input 

import os
import requests
import getpass

print("🔑 Authenticating with your Paid OpenAI Account...")
# Securely prompt for the token without showing characters on the screen
openai_key = getpass.getpass("Paste your OpenAI API Key (starts with sk-...) and press Enter: ")

if not openai_key.strip():
    raise ValueError("❌ API Key cannot be empty.")

# Enforce the strict public API server destination to bypass corporate 403 walls
API_URL = "https://api.openai.com/v1/models"
headers = {"Authorization": f"Bearer {openai_key.strip()}"}

print("\n📡 Connecting to OpenAI servers...")
# Request the official model configurations deployment list
resp = requests.get(API_URL, headers=headers)

if resp.status_code == 200:
    models_data = resp.json().get("data", [])
    
    # Categorize OpenAI models into operational groups based on naming conventions
    gpt_models = [m for m in models_data if 'gpt' in m['id']]
    embedding_models = [m for m in models_data if 'embed' in m['id']]
    other_models = [m for m in models_data if 'gpt' not in m['id'] and 'embed' not in m['id']]
    
    print(f"\n=== Chat/Generation Models (GPT): {len(gpt_models)} ===")
    for m in sorted(gpt_models, key=lambda x: x['id'])[:10]:
        print(f"  {m['id']}")
        
    print(f"\n=== Feature Extraction Models (Embeddings): {len(embedding_models)} ===")
    for m in sorted(embedding_models, key=lambda x: x['id']):
        print(f"  {m['id']}")
        
    print(f"\n=== Specialized Models (Moderation/Audio/DALL-E): {len(other_models)} ===")
    for m in sorted(other_models, key=lambda x: x['id'])[:10]:
        print(f"  {m['id']}")
else:
    print(f"❌ Could not list OpenAI endpoints. Error Code: {resp.status_code}")
    print(f"Details: {resp.text}")
    print("\nIf you still see a 403, verify that your account has a positive prepaid credit balance ($5.00) active inside the platform dashboard.")



🔑 Authenticating with your Paid OpenAI Account...


Paste your OpenAI API Key (starts with sk-...) and press Enter:  [REDACTED]


📡 Connecting to OpenAI servers...

=== Chat/Generation Models (GPT): 83 ===
  chatgpt-image-latest
  gpt-4.1
  gpt-4.1-2025-04-14
  gpt-4.1-mini
  gpt-4.1-mini-2025-04-14
  gpt-4o
  gpt-4o-2024-08-06
  gpt-4o-2024-11-20
  gpt-4o-mini
  gpt-4o-mini-2024-07-18

=== Feature Extraction Models (Embeddings): 3 ===
  text-embedding-3-large
  text-embedding-3-small
  text-embedding-ada-002

=== Specialized Models (Moderation/Audio/DALL-E): 10 ===
  chat-latest
  o3
  o3-2025-04-16
  omni-moderation-2024-09-26
  omni-moderation-latest
  tts-1
  tts-1-1106
  tts-1-hd
  tts-1-hd-1106
  whisper-1


**List the available foundational models inside the Databricks Free Edition** without using any OpenAI keys or tokens

In [0]:
# List available foundation models in Databricks Free Edition without any API keys
import mlflow.deployments

# 1. Initialize the built-in, local free tier MLflow deployment driver
client = mlflow.deployments.get_deploy_client("databricks")

print("📡 Scanning active foundation model endpoints on your free workspace shard...")

try:
    # 2. Query the local workspace driver to list out active endpoints
    # list_endpoints returns a list of dictionaries in the Databricks Free Edition runtime
    endpoints_raw = client.list_endpoints()
    
    # Safely extract the string names from the raw dictionary structure
    active_endpoints = [e.get("name") for e in endpoints_raw if isinstance(e, dict)]
    
    if active_endpoints:
        # 3. Categorize the discovered models based on standard naming conventions
        chat_models = [name for name in active_endpoints if any(kw in name.lower() for kw in ["llama", "dbrx", "mixtral", "instruct", "chat"])]
        embedding_models = [name for name in active_endpoints if any(kw in name.lower() for kw in ["gte", "bge", "embedding"])]
        other_models = [name for name in active_endpoints if name not in chat_models and name not in embedding_models]
        
        print(f"\n=== Chat / Generation Models (Instruct): {len(chat_models)} ===")
        for model in sorted(chat_models):
            print(f"  {model}")
            
        print(f"\n=== Feature Extraction Models (Embeddings): {len(embedding_models)} ===")
        for model in sorted(embedding_models):
            print(f"  {model}")
            
        if other_models:
            print(f"\n=== Specialized / Other System Endpoints: {len(other_models)} ===")
            for model in sorted(other_models):
                print(f"  {model}")
    else:
        print("\n⚠️ The model endpoint list returned empty.")
        print("This usually means no public token-serving configurations are active in this specific region shard.")

except Exception as e:
    print(f"\n❌ Local Driver Query Failed: {e}")
    print("\nExpected default foundation models pre-arranged for the associate exam:")
    print("  Chat/Instruct: databricks-meta-llama-3-1-70b-instruct")
    print("  Embedding:     databricks-gte-large-en")
    print("  Embedding:     databricks-bge-large-en")



📡 Scanning active foundation model endpoints on your free workspace shard...

=== Chat / Generation Models (Instruct): 4 ===
  databricks-llama-4-maverick
  databricks-meta-llama-3-1-8b-instruct
  databricks-meta-llama-3-3-70b-instruct
  databricks-qwen3-next-80b-a3b-instruct

=== Feature Extraction Models (Embeddings): 3 ===
  databricks-bge-large-en
  databricks-gte-large-en
  databricks-qwen3-embedding-0-6b

=== Specialized / Other System Endpoints: 4 ===
  databricks-gemma-3-12b
  databricks-gpt-oss-120b
  databricks-gpt-oss-20b
  databricks-qwen35-122b-a10b


**Decision Framework Summary**

## Agent Brick Decision Framework

| Primary Goal | Recommended Agent |
|---|---|
| Answer questions from DOCS | **KNOWLEDGE ASSISTANT** |
| Answer questions from TABLES | **AI/BI GENIE** |
| Route multiple task types to different specialists | **SUPERVISOR AGENT** |
| Extract structured fields from unstructured text | **INFORMATION EXTRACTION** |
| Parse document layout, OCR, extract tables from images | **DOCUMENT PARSING** |
| Custom text task (translate, generate, classify) | **CUSTOM LLM** |
| None of the above / need full control with OSS libraries | **CODE YOUR OWN AGENT** |

## Keywords to Watch for on the Exam

| Agent Type | Keywords |
|---|---|
| Knowledge Assistant | "knowledge base", "FAQ", "documentation", "Q&A", "answer questions from docs" |
| AI/BI Genie | "tables", "SQL", "revenue", "metrics", "structured data Q&A", "text-to-SQL" |
| Supervisor Agent | "route", "different types of requests", "multiple specialists", "delegate", "orchestrate" |
| Information Extraction | "extract fields", "structured JSON", "invoices", "contracts", "entity resolution" |
| Document Parsing | "OCR", "scanned", "table extraction", "layout analysis", "document structure" |
| Custom LLM | "translate", "generate content", "custom task", "code generation", "specialized model" |
| Code Your Own | "LangGraph", "DSPy", "custom control flow", "OSS libraries", "full flexibility" |